# Can we predict 30-day readmission (`readmit_30d`)?

**Key design point:** the data has 11,868 rows but only ~1,500 people (~8 stays each). Rows from the same person are not independent, so a random row-wise split would put the same person in train and test and could inflate scores. All evaluation here uses **person-grouped** cross-validation.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
SEED=0
df=pd.read_csv('data.csv')
print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data checks

In [ ]:
print(df.shape, '| people:', df.person_key.nunique(), '| duplicate rows:', df.duplicated().sum())
print('missing values:', int(df.isna().sum().sum()))
print('readmission rate: %.3f' % df.readmit_30d.mean())
df.describe().T

In [ ]:
g=df.groupby('person_key')
print(g.size().describe()[['min','50%','max']].to_dict())
print('duplicated (person, visit_number):', int(df.duplicated(['person_key','visit_number']).sum()))
print('persons whose visit_number is not 1..n:', int((g.visit_number.max()!=g.size()).sum()))

## 2. Univariate look
Readmission rate by feature decile (numeric) or level (categorical).

In [ ]:
num=['visit_number','birth_year','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
fig,ax=plt.subplots(3,3,figsize=(14,10)); ax=ax.ravel()
for a,col in zip(ax,num):
    b=pd.qcut(df[col],10,duplicates='drop')
    r=df.groupby(b,observed=True).readmit_30d.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(df.readmit_30d.mean(),ls='--',c='grey')
    a.set_title(col); a.set_xlabel('decile'); a.set_ylabel('readmit rate')
r=df.groupby('sex').readmit_30d.mean(); ax[-1].bar(r.index,r.values); ax[-1].set_title('sex')
plt.tight_layout(); plt.show()

## 3. Modelling setup
Features: all columns except `person_key` (an identifier). Evaluation: 5-fold `StratifiedGroupKFold` grouped by person, out-of-fold predictions; ROC-AUC and PR-AUC (base rate ≈ 0.20, so PR-AUC should be read against that). A bootstrap over people gives a CI for AUC.

In [ ]:
feats=['visit_number','birth_year','sex','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
X=df[feats]; y=df.readmit_30d.values; grp=df.person_key.values
cv=StratifiedGroupKFold(5,shuffle=True,random_state=SEED)
numf=[f for f in feats if f!='sex']
pre=ColumnTransformer([('n',StandardScaler(),numf),('c',OneHotEncoder(drop='if_binary'),['sex'])])
models={
    'baseline (prior)':DummyClassifier(strategy='prior'),
    'logistic':make_pipeline(pre,LogisticRegression(max_iter=1000)),
    'grad. boosting':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,l2_regularization=1.0,random_state=SEED),
}
Xh=X.assign(sex=(X.sex=='F').astype(int))
oof={}
for n,m in models.items():
    oof[n]=cross_val_predict(m,Xh if 'boost' in n else X,y,cv=cv,groups=grp,method='predict_proba')[:,1]
ids=np.unique(grp); idx={i:np.where(grp==i)[0] for i in ids}
def boot(p,B=300):
    rng=np.random.default_rng(SEED); out=[]
    for _ in range(B):
        s=np.concatenate([idx[i] for i in rng.choice(ids,len(ids))]); out.append(roc_auc_score(y[s],p[s]))
    return np.percentile(out,[2.5,97.5])
rows=[]
for n,p in oof.items():
    ci=boot(p) if n!='baseline (prior)' else (.5,.5)
    rows.append(dict(model=n,roc_auc=roc_auc_score(y,p),auc_95ci=f'{ci[0]:.3f}-{ci[1]:.3f}',pr_auc=average_precision_score(y,p),brier=brier_score_loss(y,p)))
pd.DataFrame(rows).round(4)

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
for n,p in oof.items():
    f,t,_=roc_curve(y,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(y,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR'); ax[0].legend()
ax[1].axhline(y.mean(),c='k',ls=':'); ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision')
for n in ['logistic','grad. boosting']:
    fp,mp=calibration_curve(y,oof[n],n_bins=10,strategy='quantile'); ax[2].plot(mp,fp,'o-',label=n)
ax[2].plot([0,.5],[0,.5],'k:'); ax[2].set(title='Calibration',xlabel='predicted',ylabel='observed'); ax[2].legend()
plt.tight_layout(); plt.show()

## 4. What drives the prediction?

In [ ]:
lr=models['logistic'].fit(X,y)
coef=pd.Series(lr[-1].coef_[0],index=numf+['sex (F vs M)']).sort_values()
print('Standardised logistic coefficients (log-odds per 1 SD):'); print(coef.round(3))
imp=np.zeros(len(feats))
for tr,te in cv.split(Xh,y,grp):
    m=models['grad. boosting'].fit(Xh.iloc[tr],y[tr])
    r=permutation_importance(m,Xh.iloc[te],y[te],scoring='roc_auc',n_repeats=10,random_state=SEED)
    imp+=r.importances_mean/5
pd.Series(imp,index=feats).sort_values().plot.barh(title='Permutation importance (AUC drop, grouped CV)'); plt.show()

## 5. Is the signal usable? Risk stratification
Readmission concentration by out-of-fold risk decile (logistic model).

In [ ]:
p=oof['logistic']
t=pd.DataFrame({'p':p,'y':y}); t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=range(1,11))
s=t.groupby('decile',observed=True).agg(n=('y','size'),pred=('p','mean'),observed=('y','mean')).round(3)
s['lift']=(s.observed/y.mean()).round(2); print(s)
top=t.p>=t.p.quantile(.8)
print(f"\nTop 20% highest-risk: observed rate {t.y[top].mean():.3f} vs base {y.mean():.3f}; captures {t.y[top].sum()/t.y.sum():.1%} of readmissions")

## 6. Does a person's own history help?
Readmission rates vary a lot between people. Adding prior-visit outcomes (earlier `visit_number`s only) as features tests whether there is a person-level effect; this is only usable in practice if prior outcomes are available at prediction time.

In [ ]:
d=df.sort_values(['person_key','visit_number']).copy()
cum=d.groupby('person_key').readmit_30d.cumsum()-d.readmit_30d
d['prior_readmits']=cum; d['prior_rate']=(cum/(d.visit_number-1).clip(lower=1)).where(d.visit_number>1,np.nan)
Xd=d[feats].assign(sex=(d.sex=='F').astype(int)); yd=d.readmit_30d.values; gd=d.person_key.values
m=HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,l2_regularization=1.0,random_state=SEED)
for name,Xi in [('without history',Xd),('with prior_readmits/prior_rate',Xd.assign(prior_readmits=d.prior_readmits,prior_rate=d.prior_rate))]:
    pp=cross_val_predict(m,Xi,yd,cv=StratifiedGroupKFold(5,shuffle=True,random_state=SEED),groups=gd,method='predict_proba')[:,1]
    print(f'{name:35s} AUC={roc_auc_score(yd,pp):.3f}  PR-AUC={average_precision_score(yd,pp):.3f}')

## 7. Conclusions (from a run with SEED=0)
* **Yes, but only modestly, from the stay/patient features alone.** Person-grouped out-of-fold ROC-AUC ≈ 0.65 (95% CI ≈ 0.63–0.66) for logistic regression vs 0.50 for the baseline; PR-AUC ≈ 0.32 vs 0.20 base rate. Gradient boosting (≈ 0.64) does not beat the linear model, so the signal looks mostly additive/linear.
* **Drivers:** longer stay, more discharge medications and higher `lab_a` raise risk; demographics, BMI, `lab_b`, postcode have small effects.
* **Practical value:** the top-risk decile has ≈ 2x the base rate (≈ 40% vs 20%); the top 20% captures ≈ 35% of readmissions. Probabilities are well calibrated. Useful for coarse prioritisation, not for confident individual prediction.
* **Person history is far more informative:** adding prior-visit readmission outcomes lifts AUC to ≈ 0.82. This suggests a strong person-level propensity not captured by the recorded features. It is only usable if earlier outcomes are known when predicting (true for repeat patients, not for first stays; first visits have no history).
* **Caveats:** one dataset, no external/temporal validation; no visit dates, so 'earlier' relies on `visit_number`; readmission to other hospitals may be missing. Don't use a row-wise random split here — it would leak person information.